# Gallstone Data Processing in Jupyter Notebook
This notebook processes the datasets, and splitting subjects into groups for further analysis.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.utils.validation import check_is_fitted
import joblib


In [ ]:
class GallstoneDataProcessor:
    def __init__(self, data_path, target_col="Gallstone Status", seed=42):
        self.data_path = data_path
        self.df = pd.read_csv(data_path, low_memory=False)
        self.target_col = target_col
        self.seed = seed
        self.pipe_ = None
        self.feature_names_ = None

        # Ensure binary numeric target 0/1 if text
        if self.df[self.target_col].dtype == object:
            mapping = {"No": 0, "Yes": 1, "no": 0, "yes": 1, "0": 0, "1": 1}
            self.df[self.target_col] = self.df[self.target_col].map(lambda x: mapping.get(str(x), x))
        if not np.issubdtype(self.df[self.target_col].dtype, np.number):
            self.df[self.target_col] = pd.Categorical(self.df[self.target_col]).codes
        self.df[self.target_col] = self.df[self.target_col].astype(int)

    def _split_xy(self):
        X = self.df.drop(columns=[self.target_col])
        y = self.df[self.target_col]
        return X, y

    def _build_preprocessor(self, X):
        cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
        num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
        pre = ColumnTransformer(
            transformers=[
                ("num", StandardScaler(), num_cols),
                ("cat", OneHotEncoder(handle_unknown="ignore", drop="if_binary"), cat_cols),
            ],
            remainder="drop",
            verbose_feature_names_out=False,
        )
        return pre, num_cols, cat_cols

    def preprocess_and_split(self, out_dir="data/processed", test_size=0.2):
        out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)

        X, y = self._split_xy()
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=test_size, stratify=y, random_state=self.seed
        )

        pre, num_cols, cat_cols = self._build_preprocessor(X_tr)
        self.pipe_ = Pipeline([("preprocessor", pre)])

        # Fit ONLY on train → no leakage
        X_tr_t = self.pipe_.fit_transform(X_tr)
        X_te_t = self.pipe_.transform(X_te)

        feat_names = self.pipe_.named_steps["preprocessor"].get_feature_names_out().tolist()
        self.feature_names_ = feat_names

        train_df = pd.DataFrame(X_tr_t, columns=feat_names)
        train_df[self.target_col] = y_tr.to_numpy()
        train_df["Set"] = "train"

        test_df = pd.DataFrame(X_te_t, columns=feat_names)
        test_df[self.target_col] = y_te.to_numpy()
        test_df["Set"] = "test"

        all_df = pd.concat([train_df, test_df], ignore_index=True)

        # Save
        train_csv = out_dir / "gallstone_train_preprocessed.csv"
        test_csv  = out_dir / "gallstone_test_preprocessed.csv"
        all_csv   = out_dir / "gallstone_all_preprocessed.csv"
        pp_path   = out_dir / "gallstone_preprocessor.joblib"

        train_df.to_csv(train_csv, index=False)
        test_df.to_csv(test_csv, index=False)
        all_df.to_csv(all_csv, index=False)
        joblib.dump(
            {
                "pipeline": self.pipe_,
                "feature_names": feat_names,
                "target_col": self.target_col,
                "seed": self.seed
            },
            pp_path
        )
        print("Saved:\n -", train_csv, "\n -", test_csv, "\n -", all_csv, "\n -", pp_path)

        return {
            "train_csv": str(train_csv),
            "test_csv": str(test_csv),
            "all_csv": str(all_csv),
            "preprocessor_joblib": str(pp_path),
            "n_features_after_encoding": len(feat_names)
        }

    def transform_new(self, raw_df: pd.DataFrame, joblib_path):
        bundle = joblib.load(joblib_path)
        pipe = bundle["pipeline"]
        feat_names = bundle["feature_names"]
        check_is_fitted(pipe)
        X_new = raw_df.drop(columns=[bundle["target_col"]], errors="ignore")
        Xt = pipe.transform(X_new)
        return pd.DataFrame(Xt, columns=feat_names)


In [ ]:
# --- REPLACE your run cell with this ---
RAW = "../data/raw/dataset-uci.csv"          # adjust if needed
OUT = "../data/processed"                     # new processed folder

processor = GallstoneDataProcessor(RAW, target_col="Gallstone Status", seed=42)
artifacts = processor.preprocess_and_split(out_dir=OUT, test_size=0.2)
artifacts
